# 01 — Split raw data & export CSV

**Mục tiêu**: tách dataset raw thành train/test 90/10 và xuất ra **CSV** theo đúng tên yêu cầu nộp bài.

**Input**:
- `data/raw/data.parquet` — dataset gốc (606,878 dòng × 14 cột)

**Output** (đặt trong `data/raw/`):
- `raw_data_train.csv` — 90% raw, dùng cho clean + train clustering
- `raw_data_test.csv`  — 10% raw, hold-out để kiểm tra pipeline

**Cấu hình**:
- `test_size = 0.1`
- `random_state = 42` (đồng bộ với các notebook khác)
- `stratify = year` — giữ phân phối các năm 2022–2026 đều ở cả train và test

## 1. Setup

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW_PATH          = Path('../data/raw/data.parquet')
OUT_TRAIN_CSV     = Path('../data/raw/raw_data_train.csv')
OUT_TEST_CSV      = Path('../data/raw/raw_data_test.csv')
OUT_TRAIN_PARQUET = Path('../data/raw/raw_data_train.parquet')
OUT_TEST_PARQUET  = Path('../data/raw/raw_data_test.parquet')

RANDOM_STATE = 42
TEST_SIZE    = 0.10
STRATIFY_COL = 'year'

## 2. Load raw

In [3]:
df = pd.read_parquet(RAW_PATH)

print(f'Loaded: {len(df):,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
print()
print('Phân phối year:')
print(df[STRATIFY_COL].value_counts().sort_index().to_string())

Loaded: 606,878 rows × 14 columns
Memory: 1217.4 MB

Phân phối year:
year
2022    111033
2023    129674
2024    159049
2025    160386
2026     46736


## 3. Stratified split 90/10

Stratify theo `year` để train/test có cùng tỉ lệ các năm — quan trọng vì dataset chứa data đa năm và mỗi năm có thể có distribution khác nhau.

In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=df[STRATIFY_COL],
)

print(f'Train: {len(train_df):,} rows  ({len(train_df) / len(df) * 100:.2f}%)')
print(f'Test : {len(test_df):,} rows  ({len(test_df) / len(df) * 100:.2f}%)')
print()

compare = pd.DataFrame({
    'full_pct' : df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
    'train_pct': train_df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
    'test_pct' : test_df[STRATIFY_COL].value_counts(normalize=True).sort_index() * 100,
}).round(2)

print('Phân phối year sau khi split (%):')
print(compare.to_string())

Train: 546,190 rows  (90.00%)
Test : 60,688 rows  (10.00%)

Phân phối year sau khi split (%):
      full_pct  train_pct  test_pct
year                               
2022     18.30      18.30     18.30
2023     21.37      21.37     21.37
2024     26.21      26.21     26.21
2025     26.43      26.43     26.43
2026      7.70       7.70      7.70


## 4. Export CSV

- `encoding='utf-8'` để giữ tiếng Việt có dấu.
- `index=False` để không thêm cột index không cần thiết.
- CSV cho 600k dòng sẽ khá lớn (~1 GB cho train) — chấp nhận được vì format được yêu cầu.

In [5]:
%%time
OUT_TRAIN_CSV.parent.mkdir(parents=True, exist_ok=True)

# Parquet — dùng cho pipeline downstream (nhanh hơn 5-10x)
train_df.to_parquet(OUT_TRAIN_PARQUET, compression='snappy', index=False)
test_df.to_parquet (OUT_TEST_PARQUET,  compression='snappy', index=False)
print(f'✓ Saved train parquet → {OUT_TRAIN_PARQUET}  ({OUT_TRAIN_PARQUET.stat().st_size / 1024**2:.1f} MB)')
print(f'✓ Saved test  parquet → {OUT_TEST_PARQUET}   ({OUT_TEST_PARQUET.stat().st_size / 1024**2:.1f} MB)')

# CSV — artifact để nộp bài (đúng yêu cầu đồ án)
train_df.to_csv(OUT_TRAIN_CSV, index=False, encoding='utf-8')
test_df.to_csv (OUT_TEST_CSV,  index=False, encoding='utf-8')
print(f'✓ Saved train CSV → {OUT_TRAIN_CSV}  ({OUT_TRAIN_CSV.stat().st_size / 1024**2:.1f} MB)')
print(f'✓ Saved test  CSV → {OUT_TEST_CSV}   ({OUT_TEST_CSV.stat().st_size / 1024**2:.1f} MB)')

✓ Saved train parquet → ..\data\raw\raw_data_train.parquet  (456.5 MB)
✓ Saved test  parquet → ..\data\raw\raw_data_test.parquet   (50.6 MB)
✓ Saved train CSV → ..\data\raw\raw_data_train.csv  (1054.0 MB)
✓ Saved test  CSV → ..\data\raw\raw_data_test.csv   (117.2 MB)
CPU times: total: 26.2 s
Wall time: 27.1 s


## 5. Sanity check

Đọc lại CSV để xác nhận shape, dtype, encoding tiếng Việt vẫn đúng.

In [6]:
train_reload = pd.read_csv(OUT_TRAIN_CSV, encoding='utf-8')
test_reload  = pd.read_csv(OUT_TEST_CSV,  encoding='utf-8')

print(f'Train reload: {train_reload.shape}  (expected {train_df.shape})')
print(f'Test  reload: {test_reload.shape}   (expected {test_df.shape})')
assert train_reload.shape == train_df.shape
assert test_reload.shape  == test_df.shape

print('\nSample 3 dòng train (kiểm tra tiếng Việt):')
print(train_reload[['job_title', 'location', 'salary']].head(3).to_string(index=False))

Train reload: (546190, 14)  (expected (546190, 14))
Test  reload: (60688, 14)   (expected (60688, 14))

Sample 3 dòng train (kiểm tra tiếng Việt):
                                                job_title                                                                                                                            location                     salary
      [Quảng Bình - Vincom Quảng Bình] Tổ Trưởng Cửa Hàng                           Vincom Quảng Bình | Tầng 26, Tòa nhà CII Tower, số 152 Điện Biên Phủ, Phường Thạnh Mỹ Tây, TP Hồ Chí Minh  7.900.000 - 8.700.000 VND
  Assistant Manager Of Administration And Human Resources        40 đường 40 KDC Vạn Phúc, phường Hiệp Bình Phước | 40 Đường số 40, Phường Hiệp Bình Phước, Thành phố Thủ Đức, TP Hồ Chí Minh                      0 VND
Nhân Viên CSKH (Tư Vấn Chốt Đơn) - Lương Từ 10Tr - Quận 2 Số 12 đường 7A, Phường An Phú | Số 15 đường 54, phường Thảo Điền | Số 6 Đường 31C, Phường An Phú, Thành phố Thủ Đức, TP Hồ Chí Minh 6.000.000 - 

## 6. Tóm tắt

**Đã làm**:
1. Load `data/raw/data.parquet` (606,878 dòng).
2. Split 90/10 stratified theo `year`, `random_state=42`.
3. Export 2 file CSV: `raw_data_train.csv`, `raw_data_test.csv` (UTF-8, không index).
4. Verify bằng cách đọc lại CSV.

**Next**: chạy `02_1_clean_data.ipynb` để clean — notebook đó cần được sửa lại để đọc từ `raw_data_train.csv` + `raw_data_test.csv` và xuất ra `clean_data_train.csv` + `clean_data_test.csv`.